# Topic: Merge and Groupby

# Import libs

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Download datasets

In [2]:
!curl -O https://raw.githubusercontent.com/jakevdp/data-USstates/master/state-population.csv
!curl -O https://raw.githubusercontent.com/jakevdp/data-USstates/master/state-areas.csv
!curl -O https://raw.githubusercontent.com/jakevdp/data-USstates/master/state-abbrevs.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 57935  100 57935    0     0   229k      0 --:--:-- --:--:-- --:--:--  229k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   835  100   835    0     0   3806      0 --:--:-- --:--:-- --:--:--  3812
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   872  100   872    0     0   4782      0 --:--:-- --:--:-- --:--:--  4765


# Define utilities

In [3]:
class display(object):
    """Display HTML representation of multiple objects"""
    table_template = \
    """
    <div style="float: left; padding: 1px;">
    <p  style='font-family:"Courier New", Courier, monospace;
        color:olive;
        font-size:120%;
        font-weight:bold;'>{0}</p>
    <hr>
    {1}
    </div>
    """
    caption_template = """\
    <div  style='
          background-color:#D6DBDF;'>
      <p  style='font-family:"Courier New", Courier, monospace;
          color:#0000FF;
          font-size:120%;
          font-weight:bold;'>
          {0}
      </p>
    </div>
    """
    def __init__(self, *args):
        self.frames = list(filter(lambda item: not item.strip().startswith('#'), args))
        self.caption = list(filter(lambda item: item.strip().startswith('#'), args))
        self.caption = [item[1:].strip().upper() for item in self.caption]
        if len(self.caption) == 0:
          self.caption = None
        else:
          self.caption = ':'.join(self.caption)

    def _repr_html_(self):
        if self.caption is not None:
          return self.caption_template.format(self.caption) + \
               '\n'.join(self.table_template.format(frame, eval(frame)._repr_html_())
                         for frame in self.frames)
        else:
          return '\n'.join(self.table_template.format(frame, eval(frame)._repr_html_())
                         for frame in self.frames)

    #def __repr__(self):
    #    return '\n\n'.join(a + '\n' + repr(eval(a))
    #                       for a in self.args)

# One-to-One

In [4]:
df1 = pd.DataFrame({'employee': ['Bob', 'Jake', 'Lisa', 'Sue', 'Mary'],
                    'group': ['Accounting', 'Engineering', 'Engineering', 'HR', 'Engineering']})
df2 = pd.DataFrame({'employee': ['Lisa', 'Bob', 'Jake', 'Sue', 'Tom', 'Anne'],
                    'hire_date': [2004, 2008, 2012, 2014, 2010, 2008]})
display('df1', 'df2')

,employee,group
0,Bob,Accounting
1,Jake,Engineering
2,Lisa,Engineering
3,Sue,HR
4,Mary,Engineering
,employee,hire_date
0,Lisa,2004
1,Bob,2008
2,Jake,2012
3,Sue,2014


## inner - default

In [7]:
pd.merge(df1, df2)

,employee,group,hire_date
0,Bob,Accounting,2008
1,Jake,Engineering,2012
2,Lisa,Engineering,2004
3,Sue,HR,2014


## inner - expicit

In [8]:
pd.merge(df1, df2, how="inner")

,employee,group,hire_date
0,Bob,Accounting,2008
1,Jake,Engineering,2012
2,Lisa,Engineering,2004
3,Sue,HR,2014


## on vs left_on and right_on

In [9]:
df1 = pd.DataFrame({'employeeA': ['Bob', 'Jake', 'Lisa', 'Sue', 'Mary'],
                    'group': ['Accounting', 'Engineering', 'Engineering', 'HR', 'Engineering']})
df2 = pd.DataFrame({'employeeB': ['Lisa', 'Bob', 'Jake', 'Sue', 'Tom', 'Anne'],
                    'hire_date': [2004, 2008, 2012, 2014, 2010, 2008]})
display('df1', 'df2')

,employeeA,group
0,Bob,Accounting
1,Jake,Engineering
2,Lisa,Engineering
3,Sue,HR
4,Mary,Engineering
,employeeB,hire_date
0,Lisa,2004
1,Bob,2008
2,Jake,2012
3,Sue,2014


In [12]:
pd.merge(df1, df2, how="inner", left_on="employeeA", right_on="employeeB").drop(["employeeB"], axis=1)

,employeeA,group,hire_date
0,Bob,Accounting,2008
1,Jake,Engineering,2012
2,Lisa,Engineering,2004
3,Sue,HR,2014


## validate the relationship

In [26]:
df1 = pd.DataFrame({'employee': ['Bob', 'Jake', 'Lisa', 'Sue', 'Mary'],
                    'group': ['Accounting', 'Engineering', 'Engineering', 'HR', 'Engineering']})
df2 = pd.DataFrame({'employee': ['Lisa', 'Bob', 'Jake', 'Sue', 'Tom', 'Anne'],
                    'hire_date': [2004, 2008, 2012, 2014, 2010, 2008]})
display('df1', 'df2')

,employee,group
0,Bob,Accounting
1,Jake,Engineering
2,Lisa,Engineering
3,Sue,HR
4,Mary,Engineering
,employee,hire_date
0,Lisa,2004
1,Bob,2008
2,Jake,2012
3,Sue,2014


In [27]:
pd.merge(df1, df2, how="inner", validate="one_to_one")

,employee,group,hire_date
0,Bob,Accounting,2008
1,Jake,Engineering,2012
2,Lisa,Engineering,2004
3,Sue,HR,2014


In [28]:
df1 = pd.DataFrame({'employee': ['Bob', 'Jake', 'Lisa', 'Sue', 'Mary'],
                    'group': ['Accounting', 'Engineering', 'Engineering', 'HR', 'Engineering']})
df3 = pd.DataFrame({'employee': ['Lisa', 'Bob', 'Jake', 'Sue', 'Tom', 'Anne', 'Bob'],
                    'hire_date': [2004, 2008, 2012, 2014, 2010, 2008, 2026]})
display('df1', 'df3')

,employee,group
0,Bob,Accounting
1,Jake,Engineering
2,Lisa,Engineering
3,Sue,HR
4,Mary,Engineering
,employee,hire_date
0,Lisa,2004
1,Bob,2008
2,Jake,2012
3,Sue,2014


In [30]:
try:
  df = pd.merge(df1, df2, how="inner", validate="one_to_one")
  print(df)

except Exception as e:
  print(e)


  employee        group  hire_date
0      Bob   Accounting       2008
1     Jake  Engineering       2012
2     Lisa  Engineering       2004
3      Sue           HR       2014


In [31]:
try:
  df = pd.merge(df1, df3, how="inner", validate="one_to_one")
  print(df)

except Exception as e:
  print(e)

Merge keys are not unique in right dataset; not a one-to-one merge


# inner => left and right

In [36]:
df1 = pd.DataFrame({'employee': ['Bob', 'Jake', 'Lisa', 'Sue', 'Mary'],
                    'group': ['Accounting', 'Engineering', 'Engineering', 'HR', 'Engineering']})
df2 = pd.DataFrame({'employee': ['Lisa', 'Bob', 'Jake', 'Sue', 'Tom', 'Anne'],
                    'hire_date': [2004, 2008, 2012, 2014, 2010, 2008]})
display('df1', 'df2')


,employee,group
0,Bob,Accounting
1,Jake,Engineering
2,Lisa,Engineering
3,Sue,HR
4,Mary,Engineering
,employee,hire_date
0,Lisa,2004
1,Bob,2008
2,Jake,2012
3,Sue,2014


In [37]:
df = pd.merge(df1, df2, how="left")
print(df)

  employee        group  hire_date
0      Bob   Accounting     2008.0
1     Jake  Engineering     2012.0
2     Lisa  Engineering     2004.0
3      Sue           HR     2014.0
4     Mary  Engineering        NaN


In [38]:
df = pd.merge(df1, df2, how="right")
print(df)

  employee        group  hire_date
0     Lisa  Engineering       2004
1      Bob   Accounting       2008
2     Jake  Engineering       2012
3      Sue           HR       2014
4      Tom          NaN       2010
5     Anne          NaN       2008


## cross

In [40]:
display("df1", "df2")

,employee,group
0,Bob,Accounting
1,Jake,Engineering
2,Lisa,Engineering
3,Sue,HR
4,Mary,Engineering
,employee,hire_date
0,Lisa,2004
1,Bob,2008
2,Jake,2012
3,Sue,2014


In [39]:
df3 = pd.merge(df1, df2, how='cross')
display('df3')

,employee_x,group,employee_y,hire_date
0,Bob,Accounting,Lisa,2004
1,Bob,Accounting,Bob,2008
2,Bob,Accounting,Jake,2012
3,Bob,Accounting,Sue,2014
4,Bob,Accounting,Tom,2010
5,Bob,Accounting,Anne,2008
6,Jake,Engineering,Lisa,2004
7,Jake,Engineering,Bob,2008
8,Jake,Engineering,Jake,2012
9,Jake,Engineering,Sue,2014


# one-to-many

In [41]:
df3 = pd.merge(df1, df2, how='inner')
df4 = pd.DataFrame({'group': ['Accounting', 'Engineering', 'HR', 'Data Analytics'],
                    'supervisor': ['Carly', 'Guido', 'Steve', 'David']})
display('df3', 'df4')

In [42]:
pd.merge(df3, df4)

,employee,group,hire_date,supervisor
0,Bob,Accounting,2008,Carly
1,Jake,Engineering,2012,Guido
2,Lisa,Engineering,2004,Guido
3,Sue,HR,2014,Steve


In [46]:
pd.merge(df3, df4)

,employee,group,hire_date,supervisor
0,Bob,Accounting,2008,Carly
1,Jake,Engineering,2012,Guido
2,Lisa,Engineering,2004,Guido
3,Sue,HR,2014,Steve


In [47]:
pd.merge(df3, df4, validate="many_to_one")

,employee,group,hire_date,supervisor
0,Bob,Accounting,2008,Carly
1,Jake,Engineering,2012,Guido
2,Lisa,Engineering,2004,Guido
3,Sue,HR,2014,Steve


# Grouping

In [50]:
students = [
    'An',
    'Mai',
    'Loan',
    'Tien'
] #assumption: name is also code
courses = [
    "Algorithms",
    "Computer Architecture",
    "Programming"
]
st_frame = pd.DataFrame(students, columns=['Student'])
co_frame = pd.DataFrame(courses, columns=['Course'])
scores = pd.merge(st_frame, co_frame, how='cross')
display('st_frame', 'co_frame', 'scores')

In [51]:
students = [
    'An',
    'Mai',
    'Loan',
    'Tien'
] #assumption: name is also code
courses = [
    "Algorithms",
    "Computer Architecture",
    "Programming"
]
st_frame = pd.DataFrame(students, columns=['Student'])
co_frame = pd.DataFrame(courses, columns=['Course'])
scores = pd.merge(st_frame, co_frame, how='cross')
scores['Score'] = pd.Series(data=np.random.randint(3, 11, (scores.shape[0])))
display('st_frame', 'co_frame', 'scores')

In [52]:
display("scores")

,Student,Course,Score
0,An,Algorithms,3
1,An,Computer Architecture,4
2,An,Programming,5
3,Mai,Algorithms,3
4,Mai,Computer Architecture,8
5,Mai,Programming,7
6,Loan,Algorithms,4
7,Loan,Computer Architecture,8
8,Loan,Programming,10
9,Tien,Algorithms,8


## aggregation

In [62]:
scores.groupby("Student").agg({"Score": ["min", "max", np.mean]}).reset_index()

/tmp/ipykernel_1659/2328954905.py:1: FutureWarning: The provided callable <function mean at 0x7f6fd7d35800> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  scores.groupby("Student").agg({"Score": ["min", "max", np.mean]}).reset_index()


Student Score              
            min max      mean
0      An     3   5  4.000000
1    Loan     4  10  7.333333
2     Mai     3   8  6.000000
3    Tien     7   8  7.666667

In [64]:
scores.groupby("Student").agg({
    "Score": ["min", "max", "mean"]
}).reset_index()

Student Score              
            min max      mean
0      An     3   5  4.000000
1    Loan     4  10  7.333333
2     Mai     3   8  6.000000
3    Tien     7   8  7.666667

In [65]:
scores.groupby("Student").agg({
    "Score": ["min", "max", "mean"],
    "Course": "count"
}).reset_index()

Student Score               Course
            min max      mean  count
0      An     3   5  4.000000      3
1    Loan     4  10  7.333333      3
2     Mai     3   8  6.000000      3
3    Tien     7   8  7.666667      3

## groupby: principle
* steps:
  1. split => groups
  2. stats for each group
  3. combine results

In [58]:
# SPLIT
for (st, group) in scores.groupby('Student'):
    print("{0:10s} shape={1}".format(st, group.shape))

An         shape=(3, 3)
Loan       shape=(3, 3)
Mai        shape=(3, 3)
Tien       shape=(3, 3)


In [61]:
type(scores.groupby('Student'))

pandas.core.groupby.generic.DataFrameGroupBy

In [60]:
# SPLIT
for (st, group) in scores.groupby('Student'):
    print("grouping value: ");
    print(st)
    print()

    print("data frame: ")
    print("type(group):")
    print(type(group))
    print()

    print("group:")
    print(group)
    print()
#

grouping value: 
An

data frame: 
type(group):
<class 'pandas.core.frame.DataFrame'>

group:
  Student                 Course  Score
0      An             Algorithms      3
1      An  Computer Architecture      4
2      An            Programming      5

grouping value: 
Loan

data frame: 
type(group):
<class 'pandas.core.frame.DataFrame'>

group:
  Student                 Course  Score
6    Loan             Algorithms      4
7    Loan  Computer Architecture      8
8    Loan            Programming     10

grouping value: 
Mai

data frame: 
type(group):
<class 'pandas.core.frame.DataFrame'>

group:
  Student                 Course  Score
3     Mai             Algorithms      3
4     Mai  Computer Architecture      8
5     Mai            Programming      7

grouping value: 
Tien

data frame: 
type(group):
<class 'pandas.core.frame.DataFrame'>

group:
   Student                 Course  Score
9     Tien             Algorithms      8
10    Tien  Computer Architecture      8
11    Tien       

## filter

In [66]:
df = scores.groupby("Student").filter(lambda subframe: subframe['Score'].mean() >= 5.0)
display('df')

,Student,Course,Score
3,Mai,Algorithms,3
4,Mai,Computer Architecture,8
5,Mai,Programming,7
6,Loan,Algorithms,4
7,Loan,Computer Architecture,8
8,Loan,Programming,10
9,Tien,Algorithms,8
10,Tien,Computer Architecture,8
11,Tien,Programming,7


In [69]:
f = scores.groupby("Student").filter(lambda subframe: subframe['Score'].mean() < 5.0)
display('df')

,Student,Course,Score
0,An,Algorithms,3
1,An,Computer Architecture,4
2,An,Programming,5


In [72]:
def filtered_by(df):
  print(type(df))
  print()
  return df['Score'].mean() < 5.0

df = scores.groupby("Student").filter(filtered_by)
display('df')

<class 'pandas.core.frame.DataFrame'>

<class 'pandas.core.frame.DataFrame'>

<class 'pandas.core.frame.DataFrame'>

<class 'pandas.core.frame.DataFrame'>



,Student,Course,Score
0,An,Algorithms,3
1,An,Computer Architecture,4
2,An,Programming,5


## transformation

In [73]:
df_new = scores.copy()
df_new['New-Score'] = scores.groupby('Student')['Score'].transform(lambda x: x + x.mean())
display('df_new')

,Student,Course,Score,New-Score
0,An,Algorithms,3,7.000000
1,An,Computer Architecture,4,8.000000
2,An,Programming,5,9.000000
3,Mai,Algorithms,3,9.000000
4,Mai,Computer Architecture,8,14.000000
5,Mai,Programming,7,13.000000
6,Loan,Algorithms,4,11.333333
7,Loan,Computer Architecture,8,15.333333
8,Loan,Programming,10,17.333333
9,Tien,Algorithms,8,15.666667


In [74]:
df_new = scores.copy()

def transformed_func(data):
  print("type(data)")
  print(type(data))
  print()
  return data + data.mean()

df_new['New-Score'] = scores.groupby('Student')['Score'].transform(transformed_func)
display('df_new')

type(data)
<class 'pandas.core.series.Series'>

type(data)
<class 'pandas.core.series.Series'>

type(data)
<class 'pandas.core.series.Series'>

type(data)
<class 'pandas.core.series.Series'>



,Student,Course,Score,New-Score
0,An,Algorithms,3,7.000000
1,An,Computer Architecture,4,8.000000
2,An,Programming,5,9.000000
3,Mai,Algorithms,3,9.000000
4,Mai,Computer Architecture,8,14.000000
5,Mai,Programming,7,13.000000
6,Loan,Algorithms,4,11.333333
7,Loan,Computer Architecture,8,15.333333
8,Loan,Programming,10,17.333333
9,Tien,Algorithms,8,15.666667


## apply

In [76]:
def add_bonus(frame):
  frame['Added'] = frame['Score'] + frame['Score'].mean()
  return frame

df_new = scores.groupby('Student').apply(add_bonus)
display('df_new')

/tmp/ipykernel_1659/4170740569.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_new = scores.groupby('Student').apply(add_bonus)


In [79]:
scores.groupby('Student')["Score"]

In [80]:
def add_bonus(data):
  return data + 1.0

s = scores.groupby('Student')["Score"].apply(add_bonus)
s

Student    
An       0      4.0
         1      5.0
         2      6.0
Loan     6      5.0
         7      9.0
         8     11.0
Mai      3      4.0
         4      9.0
         5      8.0
Tien     9      9.0
         10     9.0
         11     8.0
Name: Score, dtype: float64

# Viz